# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rabby0501/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

**Lane (kept):** Refresh / Content Opportunity Scoring. **Question:** out of many pages, which ones should a human look at first because they are losing search visibility? **Outcome:** a page is `declining` when its `trend_direction` is `down` (impressions in the last 30 days more than 20% below the 30 days before).

This notebook trains a few models and compares them with my Week-4 rule on the same pages, the same client hold-outs and the same metric (precision@K). Claims are written as observed, measured, directional or decision-support.

## 0. Setup and the Week-4 baseline

Loads the starter file and rebuilds the Week-4 slice (pages with a real position of 20 or better and at least 300 impressions in 90 days) and the Week-4 rule (`ctr_gap x log(1 + impressions)`). The check at the bottom of the cell confirms the rebuilt rule gives the same P@100 (0.720) as the Week-4 notebook, so the comparison below is against the same baseline.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isfile("data/raw/content_refresh_anonymized.csv") and os.getcwd() != "/":
        os.chdir("..")

import json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
warnings.filterwarnings("ignore")

from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance

SEED = 0
print("scikit-learn", sklearn.__version__, "| pandas", pd.__version__, "| numpy", np.__version__, "| seed", SEED)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["declining"] = (df["trend_direction"] == "down").astype(int)   # the outcome: used to train and grade, never as an input
print(f"{len(df):,} pages, {df['client_id'].nunique()} clients | overall declining rate {df['declining'].mean():.1%}")


scikit-learn 1.9.1 | pandas 3.0.5 | numpy 2.5.3 | seed 0
30,000 pages, 32 clients | overall declining rate 54.2%


In [2]:
# The SAME slice and the SAME baseline as my Week-4 notebook (w04_baseline_score), rebuilt here so the comparison is in one run.
MIN_IMPRESSIONS, MAX_POSITION = 300, 20
elig = df[(df["avg_position"] > 0) & (df["avg_position"] <= MAX_POSITION) & (df["impressions_90d"] >= MIN_IMPRESSIONS)].reset_index(drop=True)
elig["position_band"] = pd.cut(elig["avg_position"], [0, 3, 6, 10, 15, 20], labels=["1-3", "4-6", "7-10", "11-15", "16-20"])
elig["band_median_ctr"] = elig.groupby("position_band", observed=True)["ctr"].transform("median")
elig["ctr_gap"] = (1 - elig["ctr"] / elig["band_median_ctr"]).clip(0, 1)
elig["baseline_score"] = elig["ctr_gap"] * np.log1p(elig["impressions_90d"])      # my Week-4 rule, unchanged

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores), kind="stable")
    return np.asarray(labels)[order[:k]].mean()

y_all = elig["declining"].values
print(f"slice: {len(elig):,} pages from {elig['client_id'].nunique()} clients | base rate {y_all.mean():.3f}")
# consistency check against the Week-4 notebook: P@100 of the rule on the whole slice was 0.72 there.
print(f"baseline P@100 on the whole slice: {precision_at_k(elig['baseline_score'], y_all, 100):.3f}  (Week-4 notebook: 0.720)")


slice: 13,205 pages from 29 clients | base rate 0.607
baseline P@100 on the whole slice: 0.720  (Week-4 notebook: 0.720)


## 1. Method choice and why

**The question is a ranking question** ("which pages first?") with an observed yes/no outcome, so I train classifiers and use each page's predicted probability as its score. A ranking is judged at the top of the list, so the main metric is **precision@K** (K = 50, 100, 500), the same one as in Week 4, with **ROC-AUC** as a whole-list check.

**Four methods, from readable to strong, all on the same features:**

- **Logistic regression** is the readable starting point: one signed weight per feature.
- **Decision tree, depth 3** is the most readable of all: I can print it and read it. It is only three questions deep on purpose.
- **Random forest** is the stronger, less readable option for tangled signals.
- **Gradient boosting** is the "safe" boosted option from the toolkit (small learning rate, shallow trees). I use it as a second strong model, not as a default winner.

**Features (22).** Only things knowable from a page's own 90-day record: visibility (impressions, clicks, CTR, position, days with impressions, and the CTR gap from my Week-4 rule), site activity (sessions, engagement), page age and upkeep, and keyword/content context. Raw counts are `log1p`-transformed because traffic is heavy-tailed. Missing keyword or word-count values get an explicit `has_` flag instead of a silent fill with 0, because missingness follows content type.

**Left out on purpose:** the outcome columns (`trend_pct`, `trend_direction`), the six 30-day columns the trend is built from, the ids, the duplicate tier columns and the LLM provider/model columns. The cell asserts that none of the forbidden columns is among the features.

In [3]:
# Features: things knowable from a page's own 90-day record. Raw counts get log1p (traffic is heavy-tailed);
# missing keyword / word-count values get an explicit has_ flag instead of a silent fill with 0.
for c in ["impressions_90d", "clicks_90d", "sessions_90d", "search_volume", "cpc", "word_count"]:
    elig["log_" + c] = np.log1p(elig[c])
elig["has_keyword_data"] = elig["search_volume"].notna().astype(int)
elig["has_word_count"] = elig["word_count"].notna().astype(int)
elig["is_keyword_article"] = (elig["content_type"] == "keyword article").astype(int)
elig["is_feedly_article"] = (elig["content_type"] == "feedly article").astype(int)
for v in ["informational", "transactional", "commercial"]:
    elig["intent_" + v] = (elig["main_intent"] == v).astype(int)

FEATURES = [
    # search visibility
    "log_impressions_90d", "log_clicks_90d", "ctr", "avg_position", "ctr_gap", "days_with_impressions",
    # site activity
    "log_sessions_90d", "days_with_sessions", "engagement_rate",
    # page age and upkeep
    "content_age_days", "days_since_last_update",
    # keyword and content context
    "log_search_volume", "competition", "log_cpc", "log_word_count", "has_keyword_data", "has_word_count",
    "is_keyword_article", "is_feedly_article", "intent_informational", "intent_transactional", "intent_commercial",
]
EXCLUDED = {
    "trend_pct / trend_direction": "the outcome is computed from them",
    "impressions_last_30d, impressions_prev_30d, clicks_last_30d, clicks_prev_30d, sessions_last_30d, sessions_prev_30d":
        "the two 30-day windows the trend is built from",
    "content_id / client_id": "pseudonymous ids: grouping only",
    "age_tier, freshness_tier, impression_tier, position_tier ...": "duplicates of numeric columns above",
    "provider_used / model_used": "not available when deciding which page to review",
}
FORBIDDEN = ["trend_pct", "trend_direction", "impressions_last_30d", "impressions_prev_30d", "clicks_last_30d",
             "clicks_prev_30d", "sessions_last_30d", "sessions_prev_30d", "declining", "content_id", "client_id"]
assert not set(FEATURES) & set(FORBIDDEN), "a trend / outcome / id column is among the features"
print(f"{len(FEATURES)} features used. Excluded on purpose:")
for k, v in EXCLUDED.items():
    print(f"   - {k}: {v}")
print("\nmissing values per feature (before imputing):")
miss = elig[FEATURES].isna().sum()
print(miss[miss > 0].to_string() if (miss > 0).any() else "none")


22 features used. Excluded on purpose:
   - trend_pct / trend_direction: the outcome is computed from them
   - impressions_last_30d, impressions_prev_30d, clicks_last_30d, clicks_prev_30d, sessions_last_30d, sessions_prev_30d: the two 30-day windows the trend is built from
   - content_id / client_id: pseudonymous ids: grouping only
   - age_tier, freshness_tier, impression_tier, position_tier ...: duplicates of numeric columns above
   - provider_used / model_used: not available when deciding which page to review

missing values per feature (before imputing):
log_search_volume     289
competition           289
log_cpc               289
log_word_count       3990


In [4]:
# Four methods, from readable to strong. Every model gets the same features and the same splits.
# Logistic regression / tree / forest need no missing values, so a median fill is learned on the training pages only (inside the pipeline).
def make_models():
    return {
        "logistic regression": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                                             LogisticRegression(C=0.5, max_iter=2000)),
        "decision tree (depth 3)": make_pipeline(SimpleImputer(strategy="median"),
                                                 DecisionTreeClassifier(max_depth=3, min_samples_leaf=50, random_state=SEED)),
        "random forest": make_pipeline(SimpleImputer(strategy="median"),
                                       RandomForestClassifier(n_estimators=300, min_samples_leaf=20, n_jobs=-1, random_state=SEED)),
        "gradient boosting": HistGradientBoostingClassifier(max_iter=150, learning_rate=0.05, max_depth=4, random_state=SEED),
    }
print(list(make_models()))


['logistic regression', 'decision tree (depth 3)', 'random forest', 'gradient boosting']


The cell above prints the features, the exclusions and the missing-value counts. Only four features have gaps (keyword volume, competition and CPC for 289 pages, word count for 3,990 pages). The median fill for them is learned inside each training fold only, so nothing from the test clients leaks into it.

## 2. Split design

**Hold out whole clients.** A page in the test set never shares a client with a training page. That matches the real use: score pages for a client the model has not seen. A random page-level split would let the model recognise a client's habits and look better than it is.

**Repeat 20 times.** The 29 clients are very uneven (one client has about a third of the pages), so one split is mostly luck. I run 20 random splits (30% of the clients held out each time, 9 clients) with fixed seeds and report the mean and the spread. The Week-4 rule has nothing to train, so it is scored on exactly the same held-out pages.

In [5]:
# Split design: hold out WHOLE CLIENTS (a page never shares a client with a training page), 30% of clients per split,
# and repeat with 20 different random splits because the 29 clients are very uneven in size.
N_SPLITS, TEST_FRACTION = 20, 0.30
groups = elig["client_id"].values
splits = []
for seed in range(N_SPLITS):
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=TEST_FRACTION, random_state=seed).split(elig, y_all, groups))
    ok = len(set(y_all[tr])) == 2 and len(set(y_all[te])) == 2        # need both outcomes on both sides
    splits.append((seed, tr, te, ok))
good = [s for s in splits if s[3]]
print(f"{len(good)} of {N_SPLITS} splits usable")

shape = pd.DataFrame([{
    "split": s, "train_pages": len(tr), "test_pages": len(te), "test_clients": len(set(groups[te])),
    "test_base_rate": y_all[te].mean()} for s, tr, te, _ in good]).round(3)
print(shape.describe().loc[["min", "50%", "max"]].round(3).to_string())

# Sanity: no client appears on both sides of any split.
assert all(not (set(groups[tr]) & set(groups[te])) for _, tr, te, _ in good)
print("\nchecked: no client is in both train and test, in any split.")


20 of 20 splits usable


     split  train_pages  test_pages  test_clients  test_base_rate
min    0.0       4628.0      1046.0           9.0           0.515
50%    9.5       9096.0      4109.0           9.0           0.605
max   19.0      12159.0      8577.0           9.0           0.721

checked: no client is in both train and test, in any split.


All 20 splits are usable (both outcomes appear on both sides). The test sets range from about 1,000 to 8,600 pages and the declining rate in them from 51.5% to 72.1%, which is why the base rate is printed next to every result and why I judge models by how they compare with the rule on the *same* split, not by the raw numbers alone. The assertion confirms that no client is on both sides of any split.

## 3. Train + compare vs my baseline

One table: base rate, two simple comparisons, my Week-4 rule, and the four models, all on the same 20 held-out-client splits. The second table is the **paired** view: model minus rule, on each split, and how many of the 20 splits the model wins.

In [6]:
# Train every model on each split's training clients; score it on that split's held-out clients.
# The Week-4 rule has no training, so it is simply scored on the same held-out pages.
KS = (50, 100, 500)
def metrics(score, y):
    return [precision_at_k(score, y, k) for k in KS] + [roc_auc_score(y, score)]
COLS = [f"P@{k}" for k in KS] + ["AUC"]

rows = []   # (split, method, metrics...)
for seed, tr, te, _ in good:
    yt = y_all[te]
    for name, model in make_models().items():
        p = model.fit(elig.iloc[tr][FEATURES], y_all[tr]).predict_proba(elig.iloc[te][FEATURES])[:, 1]
        rows.append([seed, name] + metrics(p, yt))
    rows.append([seed, "Week-4 rule (baseline)"] + metrics(elig["baseline_score"].values[te], yt))
    rows.append([seed, "biggest pages first"] + metrics(elig["impressions_90d"].values[te], yt))
    rows.append([seed, "base rate (random picking)"] + [yt.mean()] * len(KS) + [0.5])
res = pd.DataFrame(rows, columns=["split", "method"] + COLS)

order = ["base rate (random picking)", "biggest pages first", "Week-4 rule (baseline)",
         "logistic regression", "decision tree (depth 3)", "random forest", "gradient boosting"]
mean = res.groupby("method")[COLS].mean().loc[order]
sd = res.groupby("method")[COLS].std().loc[order]
table = mean.round(3).astype(str) + " +/- " + sd.round(3).astype(str)
print(f"mean +/- standard deviation over {res['split'].nunique()} held-out-client splits")
display(table)

# Paired view: model minus the Week-4 rule, on the SAME split.
base = res[res["method"] == "Week-4 rule (baseline)"].set_index("split")[COLS]
paired = []
for name in order[3:]:
    m = res[res["method"] == name].set_index("split")[COLS]
    d = m - base
    paired.append([name] + [f"{d[c].mean():+.3f}" for c in COLS] + [f"{(d['P@50'] > 0).sum()}/{len(d)}", f"{(d['AUC'] > 0).sum()}/{len(d)}"])
paired = pd.DataFrame(paired, columns=["model minus rule"] + [f"d{c}" for c in COLS] + ["splits won (P@50)", "splits won (AUC)"])
display(paired)

Path("work/outputs").mkdir(parents=True, exist_ok=True)
_ = Path("work/outputs/w05_model_results.json").write_text(json.dumps(
    {"splits": int(res["split"].nunique()), "test_fraction": TEST_FRACTION, "seed": SEED,
     "mean": mean.round(4).to_dict(orient="index"), "sd": sd.round(4).to_dict(orient="index")}, indent=2))


mean +/- standard deviation over 20 held-out-client splits


                                       P@50            P@100            P@500  \
method                                                                          
base rate (random picking)  0.623 +/- 0.064  0.623 +/- 0.064  0.623 +/- 0.064   
biggest pages first          0.364 +/- 0.08  0.379 +/- 0.084  0.462 +/- 0.078   
Week-4 rule (baseline)      0.728 +/- 0.052  0.746 +/- 0.069  0.743 +/- 0.091   
logistic regression         0.808 +/- 0.066  0.809 +/- 0.055   0.756 +/- 0.05   
decision tree (depth 3)     0.745 +/- 0.121  0.737 +/- 0.097  0.727 +/- 0.047   
random forest                0.851 +/- 0.06  0.839 +/- 0.055   0.79 +/- 0.062   
gradient boosting           0.816 +/- 0.141  0.806 +/- 0.134  0.773 +/- 0.069   

                                        AUC  
method                                       
base rate (random picking)      0.5 +/- 0.0  
biggest pages first         0.401 +/- 0.034  
Week-4 rule (baseline)       0.589 +/- 0.03  
logistic regression         0.609 +/- 0.

          model minus rule   dP@50  dP@100  dP@500    dAUC splits won (P@50)  \
0      logistic regression  +0.080  +0.062  +0.013  +0.020             16/20   
1  decision tree (depth 3)  +0.017  -0.009  -0.016  +0.023             11/20   
2            random forest  +0.123  +0.093  +0.047  +0.045             19/20   
3        gradient boosting  +0.088  +0.059  +0.030  +0.037             16/20   

  splits won (AUC)  
0            13/20  
1            15/20  
2            16/20  
3            15/20  

**Reading the table (observed, directional).**

- The **random forest** is the best on every column: P@50 0.851 against 0.728 for the Week-4 rule (+0.123, better in 19 of 20 splits), P@100 0.839 against 0.746, P@500 0.790 against 0.743, and AUC 0.633 against 0.589 (better in 16 of 20). Average declining rate in the test sets is 0.623.
- **Logistic regression** gets most of the way with a model I can read: P@50 0.808 (+0.080, 16 of 20 splits). That is about two thirds of the forest's gain.
- **Gradient boosting** (+0.088 at P@50) is not better than the forest and is the least stable (standard deviation 0.141 at P@50).
- The **depth-3 tree** does not clearly beat the rule: +0.017 at P@50 (11 of 20 splits) and slightly below it at P@100 and P@500. It is readable, but it is not the answer here.
- "Biggest pages first" (0.364 at P@50) is worse than random picking, as in Week 4.

**What the numbers do not say.** AUC is only 0.61-0.63 for every model: over the whole list the signal is modest, and most of the gain is at the top of the list. The standard deviations (0.05 to 0.14 at P@50) are large, so I would not rank the forest above logistic regression with confidence from this table alone. The forest does win against the rule in almost every split, and that is the claim I am willing to make: a learned score ranked held-out clients' pages better than my hand rule in this snapshot. The extra complexity buys about +0.05 to +0.12 at the top of the list; it does not buy a different kind of answer.

## 4. Errors and interpretation

Four checks, each answering one question:

1. **What does the forest lean on?** Permutation importance on held-out clients (shuffle one feature, see how much AUC drops), averaged over 5 splits.
2. **Does the result survive removing groups of features?** The same forest retrained on 20 splits without page age / update date, without site activity, without any clicks or CTR column, and with visibility features only.
3. **What do the readable models say?** The printed depth-3 tree and the logistic regression coefficients.
4. **Where is the model wrong?** Out-of-fold predictions (5 folds by client), split by position band and by client, plus three concrete wrong cases.

In [7]:
# (a) What does the strongest model lean on?  Permutation importance on HELD-OUT clients:
# shuffle one feature at a time in the test pages and see how much AUC drops. Averaged over 5 splits.
imp = []
for seed, tr, te, _ in good[:5]:
    model = make_models()["random forest"].fit(elig.iloc[tr][FEATURES], y_all[tr])
    r = permutation_importance(model, elig.iloc[te][FEATURES], y_all[te], scoring="roc_auc", n_repeats=5, random_state=SEED, n_jobs=-1)
    imp.append(r.importances_mean)
imp = pd.Series(np.mean(imp, axis=0), index=FEATURES).sort_values(ascending=False)
print("average AUC drop when the feature is shuffled (random forest, held-out clients, 5 splits):")
print(imp.head(8).round(4).to_string())
print("...\nfeatures that matter almost nothing:", ", ".join(imp.tail(5).index))

# (b) Do the features that look like a client fingerprint carry the result?  Retrain without them and compare (same 20 splits).
GROUPS = {
    "all features": FEATURES,
    "without page age + days since update": [f for f in FEATURES if f not in ("content_age_days", "days_since_last_update")],
    "without site activity (sessions etc.)": [f for f in FEATURES if f not in ("log_sessions_90d", "days_with_sessions", "engagement_rate")],
    "without any clicks / CTR column (no zero-click shortcut)": [f for f in FEATURES if f not in ("log_clicks_90d", "ctr", "ctr_gap")],
    "only visibility (impressions, clicks, CTR, position)": ["log_impressions_90d", "log_clicks_90d", "ctr", "avg_position", "ctr_gap", "days_with_impressions"],
}
ab = {}
for name, cols in GROUPS.items():
    out = []
    for seed, tr, te, _ in good:
        m = make_models()["random forest"].fit(elig.iloc[tr][cols], y_all[tr])
        p = m.predict_proba(elig.iloc[te][cols])[:, 1]
        out.append(metrics(p, y_all[te]))
    ab[name] = np.mean(out, axis=0)
display(pd.DataFrame(ab, index=COLS).T.round(3))


average AUC drop when the feature is shuffled (random forest, held-out clients, 5 splits):
log_clicks_90d           0.0198
content_age_days         0.0142
avg_position             0.0112
days_with_impressions    0.0052
ctr                      0.0052
ctr_gap                  0.0040
days_with_sessions       0.0038
log_search_volume        0.0020
...
features that matter almost nothing: has_word_count, log_sessions_90d, is_feedly_article, competition, log_word_count


                                                     P@50  P@100  P@500    AUC
all features                                        0.851  0.839  0.790  0.633
without page age + days since update                0.839  0.818  0.762  0.617
without site activity (sessions etc.)               0.866  0.844  0.798  0.636
without any clicks / CTR column (no zero-click ...  0.808  0.800  0.768  0.623
only visibility (impressions, clicks, CTR, posi...  0.907  0.878  0.792  0.630

**Importance and ablations.** The top three features are `log_clicks_90d` (AUC drop 0.020), `content_age_days` (0.014) and `avg_position` (0.011); the next ones are far smaller. Many features (word count, sessions, competition) matter almost nothing.

The ablations say the result is not one feature: without any clicks or CTR column the forest still reaches P@50 0.808 and AUC 0.623, above the Week-4 rule (0.728 and 0.589). Removing page age and days since update lowers AUC from 0.633 to 0.617, so age carries some signal (it may also act as a client fingerprint, because clients differ in when their content was written). Removing site activity does not hurt (P@50 0.866), and a forest on visibility features only gets the highest top-of-list numbers (P@50 0.907, P@100 0.878) with an AUC of 0.630. I tried these feature sets on the same splits after seeing the first results, so the best-looking row is probably a little optimistic. I will not pick it as "the" model; it only says that the extra columns are not what makes the model work.

In [8]:
# (c) The two readable models, so the model-vs-rule comparison can be inspected, not just scored. Trained on all pages for display only.
Xtr, ytr = elig[FEATURES], y_all
tree = make_models()["decision tree (depth 3)"].fit(Xtr, ytr)
print(export_text(tree[-1], feature_names=FEATURES, max_depth=3))

lr = make_models()["logistic regression"].fit(Xtr, ytr)
coef = pd.Series(lr[-1].coef_[0], index=FEATURES).sort_values()
print("logistic regression: standardised coefficients (positive = more likely declining)")
print(pd.concat([coef.head(5), coef.tail(5)]).round(3).to_string())


|--- log_clicks_90d <= 3.02
|   |--- content_age_days <= 312.50
|   |   |--- days_with_impressions <= 24.50
|   |   |   |--- class: 0
|   |   |--- days_with_impressions >  24.50
|   |   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- avg_position <= 4.55
|   |   |   |--- class: 1
|   |   |--- avg_position >  4.55
|   |   |   |--- class: 1
|--- log_clicks_90d >  3.02
|   |--- content_age_days <= 326.50
|   |   |--- log_clicks_90d <= 5.40
|   |   |   |--- class: 1
|   |   |--- log_clicks_90d >  5.40
|   |   |   |--- class: 0
|   |--- content_age_days >  326.50
|   |   |--- avg_position <= 6.75
|   |   |   |--- class: 0
|   |   |--- avg_position >  6.75
|   |   |   |--- class: 0

logistic regression: standardised coefficients (positive = more likely declining)
log_clicks_90d           -0.571
intent_informational     -0.301
intent_transactional     -0.266
content_age_days         -0.249
intent_commercial        -0.217
days_since_last_update    0.085
days_with_sessions   

**The readable models.** The tree splits first on `log_clicks_90d` (a cut around 20 clicks in 90 days), then on `content_age_days` (about 310-330 days), `days_with_impressions` and `avg_position`. In the logistic regression the strongest weight is `log_clicks_90d` (-0.571: more clicks, less likely to be declining), then `content_age_days` (-0.249: older pages are less likely to be declining) and the three intent flags, which are measured against pages with no intent label and should not be over-read. Both readable models lean on the same two ideas as the forest, which makes the forest's importance ranking more believable.

In [9]:
# (d) Where is the model wrong?  Out-of-fold predictions: 5 folds, each fold's clients are predicted by a forest trained on the other clients.
oof = np.zeros(len(elig))
for tr, te in GroupKFold(n_splits=5).split(elig, y_all, groups):
    oof[te] = make_models()["random forest"].fit(elig.iloc[tr][FEATURES], y_all[tr]).predict_proba(elig.iloc[te][FEATURES])[:, 1]
elig["oof_prob"] = oof
print(f"out-of-fold AUC (all clients, each predicted by a model that never saw them): {roc_auc_score(y_all, oof):.3f}")

# by position band
band = (elig.groupby("position_band", observed=True)
        .apply(lambda g: pd.Series({"n": len(g), "declining_rate": g["declining"].mean(),
                                    "AUC": roc_auc_score(g["declining"], g["oof_prob"]) if g["declining"].nunique() == 2 else np.nan}))
        .round(3))
display(band)

# by client: does the model get a client's overall level right, and does it rank pages WITHIN a client?
cl = (elig.groupby("client_id")
      .apply(lambda g: pd.Series({"n": len(g), "declining_rate": g["declining"].mean(), "mean_predicted": g["oof_prob"].mean(),
                                  "within_client_AUC": roc_auc_score(g["declining"], g["oof_prob"]) if (g["declining"].nunique() == 2 and len(g) >= 100) else np.nan}))
      .sort_values("n", ascending=False))
big = cl.head(6).round(3)
print("six biggest clients (pseudonyms):")
display(big)
print(f"clients where |mean predicted - actual declining rate| > 0.15: {(abs(cl['mean_predicted'] - cl['declining_rate']) > 0.15).sum()} of {len(cl)}")
print(f"median within-client AUC (clients with >= 100 pages): {cl['within_client_AUC'].median():.3f}")


out-of-fold AUC (all clients, each predicted by a model that never saw them): 0.622


                    n  declining_rate    AUC
position_band                               
1-3             507.0           0.750  0.798
4-6            3140.0           0.603  0.674
7-10           4507.0           0.583  0.630
11-15          3037.0           0.608  0.548
16-20          2014.0           0.631  0.545

six biggest clients (pseudonyms):


                        n  declining_rate  mean_predicted  within_client_AUC
client_id                                                                   
client_19581e27de  4677.0           0.549           0.611              0.630
client_6208ef0f77  1664.0           0.624           0.690              0.594
client_4e07408562  1581.0           0.481           0.539              0.618
client_3fdba35f04  1193.0           0.807           0.614              0.665
client_7f2253d7e2   630.0           0.959           0.579              0.628
client_f369cb89fc   570.0           0.674           0.668              0.643

clients where |mean predicted - actual declining rate| > 0.15: 15 of 29
median within-client AUC (clients with >= 100 pages): 0.628


In [10]:
# (e) Three concrete wrong cases: pages the forest was MOST sure were declining, but were not (outcome used only to find them).
top = elig.sort_values("oof_prob", ascending=False).head(100)
print(f"top-100 by predicted probability: {top['declining'].mean():.2f} actually declining (base rate {y_all.mean():.2f}); {top['client_id'].nunique()} clients")
# Is there a pattern in the confident misses?
zero_click = top["clicks_90d"] == 0
print(f"in that top 100: {int(zero_click.sum())} pages have zero clicks in 90 days -> {top[zero_click]['declining'].mean():.2f} declining; "
      f"the other {int((~zero_click).sum())} -> {top[~zero_click]['declining'].mean():.2f} declining")
print(f"zero-click pages in the whole slice: {(elig['clicks_90d'] == 0).sum():,} ({(elig['clicks_90d'] == 0).mean():.1%}); declining rate {elig[elig['clicks_90d'] == 0]['declining'].mean():.2f}; median impressions {elig[elig['clicks_90d'] == 0]['impressions_90d'].median():,.0f}")
wrong = top[top["declining"] == 0].head(3)
cols = ["client_id", "oof_prob", "impressions_90d", "avg_position", "ctr", "ctr_gap", "days_since_last_update", "content_age_days", "trend_direction", "trend_pct"]
display(wrong[cols].round(3))
print("client decline rate for those three clients:",
      ", ".join(f"{c}: {cl.loc[c, 'declining_rate']:.2f}" for c in wrong["client_id"]))


top-100 by predicted probability: 0.82 actually declining (base rate 0.61); 10 clients
in that top 100: 97 pages have zero clicks in 90 days -> 0.81 declining; the other 3 -> 1.00 declining
zero-click pages in the whole slice: 1,825 (13.8%); declining rate 0.76; median impressions 665


              client_id  oof_prob  impressions_90d  avg_position  ctr  \
9649  client_d029fa3a95     0.946              781           9.1  0.0   
1321  client_19581e27de     0.941             1058          10.0  0.0   
7694  client_d029fa3a95     0.941             1064           8.5  0.0   

      ctr_gap  days_since_last_update  content_age_days trend_direction  \
9649      1.0                      20               172              up   
1321      1.0                      20               119              up   
7694      1.0                      20               172          stable   

      trend_pct  
9649       24.0  
1321       32.2  
7694       -0.4  

client decline rate for those three clients: client_d029fa3a95: 0.60, client_19581e27de: 0.55, client_d029fa3a95: 0.60


**Where the model is wrong (observed).**

- **By position band:** the model ranks well near the top of Google and poorly further down. AUC is 0.80 for positions 1-3 (507 pages), 0.67 for 4-6, 0.63 for 7-10, and only about 0.55 for 11-15 and 16-20. Most of the pages (4,507 and 3,140) sit in the middle bands, where it is only modestly better than a coin flip.
- **By client:** the model cannot know a client's overall level. For 15 of 29 clients the average predicted probability is more than 0.15 away from the client's real declining rate (for example one client has 96% declining pages and gets an average prediction of 0.58). Within a client, the median AUC is 0.628, so it does rank a client's own pages a little better than chance. Both facts matter for a real queue: it is a better *within-client* sorter than a *between-client* judge.
- **The zero-click shortcut.** Of the 100 pages the forest is most sure about, 97 have zero clicks in 90 days, and 81% of them are declining (the other three, 100%). Across the whole slice, the 1,825 zero-click pages (13.8%) decline 76% of the time against 61% overall. So "no clicks" is a real signal, but it is also the model's easiest rule, and it is partly circular: `clicks_90d` is a 90-day total that includes the very 30 days the outcome is built from.
- **Three confident wrong cases.** The three most-confident misses all have no clicks and 781-1,064 impressions (close to my 300-impression floor), a position around 8.5-10, an update 20 days ago and an age of 119-172 days. Two of the three were actually growing (+24% and +32% impressions) and one was flat (-0.4%). They are hard because a page with a few hundred impressions and no clicks looks identical to a page that is quietly dying; with so few events, the same numbers happen on pages that are about to grow. Their clients decline 55-60% of the time, so the clients themselves are not the explanation.

**Limits I want a reader to keep in mind.** (1) The inputs are 90-day totals and the outcome is a 30-day trend inside the same 90 days, so this is a snapshot, not a clean "past predicts future" test, and the click features may be partly mechanical. Week 6 should re-cut this with a true time split. (2) I compared four feature sets on the same splits, so the best row is optimistic. (3) Twenty-nine clients, a few of them large, in one snapshot. (4) Results depend on library versions; the notebook prints them and uses fixed seeds, and tree-ensemble numbers can shift a few points between versions.

**Decision-support summary.** On held-out clients a random forest ranked pages better than my Week-4 rule (about +0.12 at P@50, 19 of 20 splits), and logistic regression gave about two thirds of that gain in a form I can read. The gain is concentrated at the top of the list and in pages ranked near the top of Google, and the model leans most on a clicks signal that overlaps the outcome window. I would treat it as a better first-pass sorter, not as a forecast.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.